In [ ]:
import torch
import torch.nn as nn

# Normalization Layer
- Position: Conv -> **Norm** -> Activation -> Pool
- Why: stable scaling for deep network. Ensure faster converge, more robust to initial setup and input
- Normalize based on **axes** used in computing mean and variance
- If `affine=False`, Norm no longer storing scale and bias, therefore converting input to mean 0 and standard deviation of 1: x_new = (x - mean)/deviation
- If `affine=True`: y = scale * x_new + bias (preserve lines)
- Preserve input shape

In [ ]:
# Norm over B H W for each C, batch dependent
# Have regularization effect, can be used as a substitute for dropout
# Useful: large batch size
# Parameters: C + C (one scale and one bias)
# train(): use mean and variance of current batch, update running mean and variance
# eval(): use stored running mean and variance, no longer batch dependent
class BatchNorm2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.batch_norm = nn.BatchNorm2d(num_features=16)

    def forward(self, x):
        return self.batch_norm(x)

In [ ]:
# Norm over H W for each B C, batch independent
# Remove instance-specific contrast information, but preserve channel structure
# Useful: style transformation, image generation (constrast norm of each image is vital)
# Parameters: C + C (one scale and one bias)
class InstanceNorm2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.instance_norm = nn.InstanceNorm2d(num_features=16)

    def forward(self, x):
        return self.instance_norm(x)

In [ ]:
# Norm over features dimension for each sample, batch independent
# Do not have regularization effect, need dropout
# Useful: Token-based models
# normalized_shape=(C, H, W) for all features in NCHW tensors,
# normalized_shape=(D,) for each token in BND tensors (sequence of tokens)
# Parameters: 2 x C x H x W (one scale and one bias for each feature)
# Or 2 x D (one scale and one bias for each token)
class LayerNorm2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = nn.LayerNorm(normalized_shape=[16, 32, 32])

    def forward(self, x):
        return self.layer_norm(x)

In [ ]:
# Norm over H, W for a group of channels, batch independent
# Bridge between layer norm and instance norm, best for small batch siz
# Useful: small batch size, detection and segmentation
# Split C into G groups, each group has C/G channels, norm over H W for each group => C/G x H x W for each sample
# Parameters: 2 x C (one scale and one bias for each channel)
class GroupNorm2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.group_norm = nn.GroupNorm(num_groups=4, num_channels=16)

    def forward(self, x):
        return self.group_norm(x)